# Домашнее задание 5
## Околокруговое приближение. Алгоритмы перехода. Базис-вектор.

**Введение в маневрирование КА · К лекции 6**  
**Срок сдачи: 2 ноября 2026 года, 23:59.**

Решите задачи перехода между близкими орбитами в околокруговом приближении:
исследуйте семейство компланарных переходов и узловой пространственный переход.

**Задания 1–3 обязательны; задание 4* — по желанию.**

### Как работать с ноутбуком

Выполняйте ячейки по порядку. Нужны Python 3, NumPy и Matplotlib.
Готовые функции выводят таблицы, строят графики и позволяют при желании проверить ответы;
их код менять не требуется. Самостоятельно получите формулы, заполните
переменные со значением `None` и текстовые ячейки с ответами.

**Чекеры предназначены только для самопроверки. Использовать их не обязательно.**
Ячейки с их вызовами можно пропустить; сохранять результаты проверок не нужно.

**Единицы и порядок компонент:**
- все углы в коде — в **радианах**; места импульсов сначала задавайте в $[0,2\pi)$;
- `phi_2`: массив формы `(3, 2)` — три перехода, по два угла в каждом;
- `dv_hat_2`: массив формы `(3, 2, 3)` — те же переходы и импульсы;
- `phi_2_chronological`: массив формы `(3, 2)` с углами в хронологическом порядке;
  к углу второго импульса при необходимости добавьте $2\pi$;
- `phi_3`: массив формы `(2,)`, `dv_hat_3`: массив формы `(2, 3)`;
- последний индекс `dv_hat_2`, `dv_hat_3` — **уже нормированные** компоненты
  $(\Delta\widehat V_r,\Delta\widehat V_t,\Delta\widehat V_z)$;
  повторно делить их на $V_0$ не нужно;
- `lambdas_2`, `lambdas_3`: пять чисел в порядке $(\lambda_1,\ldots,\lambda_5)$;
  в задаче перехода $\lambda_6=0$;
- $J$ безразмерен; в таблицах углы переводятся в градусы, импульсы и
  $\Delta V_\Sigma=V_0J$ — в м/с.

Не округляйте промежуточные результаты. Успешные численные проверки
не заменяют выводы, доказательства и объяснения. Пока ответы не заполнены,
ячейки выводят напоминания и не прерывают выполнение ноутбука.

## Задание 1. Чтение методички

Прочитайте разделы **3.1, 3.2.2 и 3.3.1**: постановку задачи перехода,
алгоритм для непересекающихся компланарных орбит и узловое решение.
Повторите достаточные условия оптимальности с базис-вектором.
Остальные пространственные случаи просмотрите на уровне областей применимости
и классификации алгоритмов; прочитайте порядок расчёта в разделе **3.4**.

Дайте краткие ответы без подробного вывода формул:

1. Почему для непересекающихся компланарных орбит место первого импульса
   можно выбирать свободно? Как это связано с годографом базис-вектора?
2. В методичке приведены два условия применимости узлового решения.
   Какое из них обеспечивает неотрицательность модулей импульсов,
   а какое — ограничение модуля базис-вектора на всём обороте?

**Ответ на вопрос 1:**

*Кратко объясните связь свободы выбора места импульса с годографом.*

**Ответ на вопрос 2:**

*Укажите смысл каждого из двух условий применимости узлового решения.*


## Общие исходные данные

$$\mu=398600{,}44158\ \text{км}^3/\text{с}^2,\qquad r_0=7000\ \text{км}.$$

Используйте обозначения и нормировку из методички.
Начало маневрирования, длительность перехода и конечное положение
аппарата на орбите свободны.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

ACCENT = "#187AA5"
SECOND = "#A36540"
INK = "#302B28"
fonts = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": "PT Serif" if "PT Serif" in fonts else "DejaVu Serif",
    "font.size": 11,
    "text.color": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})

In [ ]:
MU = 398600.44158       # км³/с²
R0 = 7000.0            # км
V0 = np.sqrt(MU / R0)  # км/с
TWO_PI = 2 * np.pi

DA_2, DEX_2, DEY_2, DI_2 = 0.004, 0.001, 0.001, 0.0
DA_3, DEX_3, DEY_3, DI_3 = 0.001, 0.0024, 0.0006, 0.0018
FIRST_ANGLES_2 = np.radians([0.0, 120.0, 240.0])
TARGET_2 = np.array([DA_2, DEX_2, DEY_2, 0.0, DI_2])
TARGET_3 = np.array([DA_3, DEX_3, DEY_3, 0.0, DI_3])
DI_MIN_STAR, DI_MAX_STAR = 0.0003, 0.0018

print(f"V₀ = {V0:.8f} км/с")
print(f"Задание 2: Δa = {DA_2}; Δeₓ = {DEX_2}; Δeᵧ = {DEY_2}; Δi = {DI_2}")
print(f"Задание 3: Δa = {DA_3}; Δeₓ = {DEX_3}; Δeᵧ = {DEY_3}; Δi = {DI_3} рад")

### Готовые функции для таблиц, графиков и самопроверки

Выполните следующие ячейки. Разбирать их код для выполнения задания не нужно.

`orbit_changes(phi, dv_hat)` возвращает вклад каждого импульса в пять левых частей
граничных условий: столбцы соответствуют $\Delta a,\Delta e_x,\Delta e_y,0,\Delta i$.
Для полного перехода сложите строки. `boundary_table(phi, dv_hat, target)`
выводит эти суммы и невязки относительно заданной правой части.

`impulse_table(phi, dv_hat, title, together=False)` печатает углы,
три компоненты и модули. При `together=True` также выводит $J$ и
суммарную характеристическую скорость в м/с. Функции принимают **один переход**:
`phi` формы `(N,)`, `dv_hat` формы `(N, 3)`.

`primer(phi, lambdas)` возвращает три компоненты $(s_r,s_t,s_z)$ базис-вектора.
`plot_primer(phi_impulses, lambdas, title)` строит его пространственный годограф
с единичной сферой и график модуля за оборот. Компоненты годографа — местные
орбитальные компоненты, а не координаты в неподвижных пространственных осях.

`check_task2(...)`, `check_task3(...)`, `check_star(...)` возвращают `True`
при успешной численной проверке и `False`, если обнаружена ошибка.
Если ответы ещё не заполнены, возвращается `None`.
Сообщения поясняют, какое условие стоит проверить.

In [ ]:
def orbit_changes(phi, dv_hat):
    """Вклады импульсов в пять граничных условий околокруговой задачи перехода."""
    phi = np.asarray(phi, dtype=float)
    dv = np.asarray(dv_hat, dtype=float)
    vr, vt, vz = dv[:, 0], dv[:, 1], dv[:, 2]
    return np.column_stack((2 * vt,
                            vr * np.sin(phi) + 2 * vt * np.cos(phi),
                            -vr * np.cos(phi) + 2 * vt * np.sin(phi),
                            vz * np.sin(phi), vz * np.cos(phi)))


def primer(phi, lambdas):
    """Базис-вектор задачи перехода, λ₆ = 0; последний индекс: r,t,z."""
    phi = np.asarray(phi, dtype=float)
    l1, l2, l3, l4, l5 = np.asarray(lambdas, dtype=float)
    return np.stack((l3 * np.sin(phi) - l2 * np.cos(phi),
                     2 * l1 + 2 * l3 * np.cos(phi) + 2 * l2 * np.sin(phi),
                     l4 * np.sin(phi) + l5 * np.cos(phi)), axis=-1)


def impulse_table(phi, dv_hat, title, together=False):
    phi = np.asarray(phi, dtype=float)
    dv = np.asarray(dv_hat, dtype=float)
    dimensional = dv * V0 * 1000
    norms = np.linalg.norm(dimensional, axis=1)
    print(title)
    print(f"{'№':>3} {'φ, град':>13} {'ΔVᵣ, м/с':>15} {'ΔVₜ, м/с':>15} "
          f"{'ΔV_z, м/с':>15} {'|ΔV|, м/с':>15}")
    for j, (angle, components, norm) in enumerate(zip(phi, dimensional, norms), 1):
        print(f"{j:3d} {np.degrees(angle):13.6f} {components[0]:15.6f} "
              f"{components[1]:15.6f} {components[2]:15.6f} {norm:15.6f}")
    if together:
        print(f"J = {np.linalg.norm(dv, axis=1).sum():.10f}")
        print(f"Суммарная характеристическая скорость: {norms.sum():.6f} м/с")


def boundary_table(phi, dv_hat, target):
    total = orbit_changes(phi, dv_hat).sum(axis=0)
    target = np.asarray(target, dtype=float)
    names = ("Δa", "Δeₓ", "Δeᵧ", "Σ V̂z sinφ", "Δi")
    print(f"{'Условие':>13} {'Сумма':>16} {'Правая часть':>16} {'Невязка':>13}")
    for name, value, desired in zip(names, total, target):
        print(f"{name:>13} {value:16.10f} {desired:16.10f} {value - desired:13.2e}")


def plot_primer(phi_impulses, lambdas, title="Базис-вектор задачи перехода"):
    phi = np.linspace(0.0, TWO_PI, 1201)
    s = primer(phi, lambdas)
    norm = np.linalg.norm(s, axis=-1)
    phi_impulses = np.asarray(phi_impulses, dtype=float) % TWO_PI
    s_impulses = primer(phi_impulses, lambdas)
    impulse_norms = np.linalg.norm(s_impulses, axis=-1)
    fig = plt.figure(figsize=(12, 5.1), constrained_layout=True)
    ax3d = fig.add_subplot(1, 2, 1, projection="3d")
    ax = fig.add_subplot(1, 2, 2)
    u = np.linspace(0, TWO_PI, 25)
    v = np.linspace(0, np.pi, 13)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    ax3d.plot_wireframe(x, y, z, color=INK, alpha=0.13, linewidth=0.55)
    ax3d.plot(s[:, 0], s[:, 1], s[:, 2], color=ACCENT, linewidth=2.1)
    ax3d.scatter(s_impulses[:, 0], s_impulses[:, 1], s_impulses[:, 2],
                 color=SECOND, s=35, depthshade=False)
    for j, vector in enumerate(s_impulses, 1):
        ax3d.plot([0, vector[0]], [0, vector[1]], [0, vector[2]],
                  color=SECOND, alpha=0.65, linewidth=1)
        ax3d.text(*(vector * 1.09), str(j), color=INK, fontsize=10)
    limit = max(1.15, float(np.abs(s).max()) * 1.15)
    ax3d.set(xlabel=r"$s_r$", ylabel=r"$s_t$", zlabel=r"$s_z$",
             xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit),
             title="Годограф и единичная сфера")
    ax3d.set_box_aspect((1, 1, 1))
    ax3d.view_init(elev=22, azim=-55)
    ax.plot(np.degrees(phi), norm, color=ACCENT, linewidth=2,
            label=r"$\|\boldsymbol{s}(\varphi)\|$")
    ax.axhline(1.0, color=SECOND, linestyle="--", label="Единичный модуль")
    if phi_impulses.size:
        ax.scatter(np.degrees(phi_impulses), impulse_norms, color=SECOND,
                   zorder=4, label="Углы приложения импульсов")
    for j, (angle, value) in enumerate(zip(phi_impulses, impulse_norms), 1):
        ax.annotate(str(j), (np.degrees(angle), value), xytext=(5, 7),
                    textcoords="offset points", color=INK)
    ax.set(xlabel=r"Угол $\varphi$, град", ylabel="Модуль базис-вектора", xlim=(0, 360))
    ax.set_ylim(0, max(1.1, float(norm.max()) * 1.1))
    ax.set_xticks(np.arange(0, 361, 60))
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    fig.suptitle(title)
    return fig, (ax3d, ax)

In [ ]:
# Простые численные проверки
_BOUNDARY_ATOL = 1e-8


def _report(title, checks):
    print(title)
    for passed, message in checks:
        print(("[OK] " if passed else "[Проверьте] ") + message)
    passed = all(bool(item[0]) for item in checks)
    print("Численные проверки пройдены." if passed
          else "Есть замечания; исправьте отмеченные пункты и повторите проверку.")
    return passed


def _maneuver_arrays(phi, dv_hat):
    try:
        phi = np.asarray(phi, dtype=float)
        dv = np.asarray(dv_hat, dtype=float)
    except (ValueError, TypeError):
        print("Углы и компоненты должны быть числовыми массивами без None.")
        return None
    if phi.shape != (2,) or dv.shape != (2, 3):
        print("Нужны два угла и массив формы (2, 3): строки — импульсы, столбцы — r,t,z.")
        return None
    if not np.all(np.isfinite(phi)) or not np.all(np.isfinite(dv)):
        print("В ответе есть NaN или бесконечность. Проверьте вычисления.")
        return None
    if np.any(phi < 0) or np.any(phi >= TWO_PI):
        print("Основные углы задавайте в радианах в [0, 2π).")
        return None
    return phi, dv


def _lambda_array(lambdas):
    try:
        l = np.asarray(lambdas, dtype=float)
    except (TypeError, ValueError):
        print("Множители должны быть пятью числами: λ₁, …, λ₅.")
        return None
    if l.shape != (5,) or not np.all(np.isfinite(l)):
        print("Нужен массив из пяти конечных чисел: [λ₁, λ₂, λ₃, λ₄, λ₅].")
        return None
    return l


def _global_norm_bound(lambdas):
    # s = c + p cosφ + q sinφ. Спектр матрицы Грама даёт точный
    # максимум ||p cosφ + q sinφ|| на полном обороте без угловой сетки.
    # Добавление ||c|| даёт верхнюю оценку; при c=0 или p=q=0 она точна.
    l1, l2, l3, l4, l5 = lambdas
    p = np.array([-l2, 2 * l3, l5])
    q = np.array([l3, 2 * l2, l4])
    gram = np.array([[p @ p, p @ q], [p @ q, q @ q]])
    return 2 * abs(l1) + np.sqrt(max(0.0, float(np.linalg.eigvalsh(gram)[-1])))


def _certificate_checks(phi, dv, lambdas):
    norm = np.linalg.norm(dv, axis=1)
    s = primer(phi, lambdas)
    nonzero = bool(np.all(norm > 1e-12))
    directions_ok = nonzero and np.allclose(s, dv / norm[:, None], rtol=0, atol=2e-5)
    bound = _global_norm_bound(lambdas)
    return [
        (nonzero, "Оба импульса ненулевые; их направления определены."),
        (directions_ok, "Общий базис-вектор сонаправлен каждому импульсу; учтены знаки компонент."),
        (np.allclose(np.linalg.norm(s, axis=1), 1, rtol=0, atol=2e-5),
         "В точках импульсов модуль базис-вектора равен единице."),
        (bound <= 1 + 1e-6,
         f"Глобальная верхняя оценка модуля на полном обороте: {bound:.8f} ≤ 1."),
    ]


def check_task2(phi, dv_hat, lambdas, phi_chronological):
    """Три двухимпульсных перехода и один общий базис-вектор."""
    if phi is None or dv_hat is None or lambdas is None or phi_chronological is None:
        print("Задание 2: сначала заполните phi_2, dv_hat_2, lambdas_2 и phi_2_chronological.")
        return None
    try:
        phi = np.asarray(phi, dtype=float)
        dv = np.asarray(dv_hat, dtype=float)
        chronological = np.asarray(phi_chronological, dtype=float)
    except (TypeError, ValueError):
        print("Углы и компоненты должны быть числовыми массивами без None.")
        return False
    if phi.shape != (3, 2) or dv.shape != (3, 2, 3) or chronological.shape != (3, 2):
        print("Нужны формы phi_2 (3, 2), dv_hat_2 (3, 2, 3), phi_2_chronological (3, 2).")
        return False
    l = _lambda_array(lambdas)
    if l is None:
        return False
    checks, costs = [], []
    for k in range(3):
        data = _maneuver_arrays(phi[k], dv[k])
        if data is None:
            return False
        angles, impulses = data
        first_error = abs(np.angle(np.exp(1j * (angles[0] - FIRST_ANGLES_2[k]))))
        checks.append((first_error <= 1e-8, f"Переход {k + 1}: заданный угол первого импульса."))
        checks.append((np.all(np.abs(impulses[:, [0, 2]]) <= _BOUNDARY_ATOL),
                       f"Переход {k + 1}: импульсы чисто трансверсальные."))
        checks.append((np.all(impulses[:, 1] > 0),
                       f"Переход {k + 1}: знаки трансверсальных компонент согласованы с Δa."))
        total = orbit_changes(angles, impulses).sum(axis=0)
        for j, name in enumerate(("Δa", "Δeₓ", "Δeᵧ")):
            error = abs(total[j] - TARGET_2[j])
            checks.append((error <= _BOUNDARY_ATOL,
                           f"Переход {k + 1}: условие на {name}; невязка {error:.2e}."))
        congruent = np.allclose(np.exp(1j * chronological[k]), np.exp(1j * angles), rtol=0, atol=1e-8)
        ordered = (np.all(np.isfinite(chronological[k]))
                   and abs(chronological[k, 0] - angles[0]) <= 1e-8
                   and 0 < chronological[k, 1] - chronological[k, 0] <= TWO_PI + 1e-8)
        checks.append((congruent and ordered,
                       f"Переход {k + 1}: углы согласованы по модулю 2π и записаны в хронологическом порядке."))
        for passed, message in _certificate_checks(angles, impulses, l):
            checks.append((passed, f"Переход {k + 1}: " + message))
        costs.append(np.linalg.norm(impulses, axis=1).sum())
    checks.append((np.ptp(costs) <= _BOUNDARY_ATOL, "Суммы модулей трёх переходов совпадают."))
    return _report("Самопроверка задания 2", checks)


def check_task3(phi, dv_hat, lambdas):
    """Один узловой переход и общий для него базис-вектор."""
    if phi is None or dv_hat is None or lambdas is None:
        print("Задание 3: сначала заполните phi_3, dv_hat_3 и lambdas_3.")
        return None
    data = _maneuver_arrays(phi, dv_hat)
    l = _lambda_array(lambdas)
    if data is None or l is None:
        return False
    phi, dv = data
    angle_error = np.abs(np.angle(np.exp(1j * (phi - np.array([0.0, np.pi])))))
    checks = [(np.all(angle_error <= 1e-8), "Углы соответствуют восходящему и нисходящему узлам, в этом порядке."),
              (abs(l[0]) <= 1e-8, "В узловом решении λ₁ = 0.")]
    total = orbit_changes(phi, dv).sum(axis=0)
    for j, name in enumerate(("Δa", "Δeₓ", "Δeᵧ", "Σ V̂z sinφ = 0", "Δi")):
        error = abs(total[j] - TARGET_3[j])
        checks.append((error <= _BOUNDARY_ATOL,
                       f"Суммарное условие на {name}; невязка {error:.2e}."))
    checks.extend(_certificate_checks(phi, dv, l))
    return _report("Самопроверка задания 3", checks)

In [ ]:
def plot_costs_star(nc_cost_function, sc_cost_function, case_function):
    """Кривые J_NC, J_s и стоимость применимого алгоритма; функции скалярные."""
    di = np.linspace(DI_MIN_STAR, DI_MAX_STAR, 401)
    try:
        nc = np.array([nc_cost_function(float(value)) for value in di], dtype=float)
        sc = np.array([sc_cost_function(float(value)) for value in di], dtype=float)
        cases = np.array([case_function(float(value)) for value in di])
    except (TypeError, ValueError) as error:
        print(f"Проверьте три функции задания 4*: {type(error).__name__}: {error}")
        return None
    if not np.all(np.isfinite(nc)) or not np.all(np.isfinite(sc)):
        print("Сначала заполните функции стоимости конечными числовыми значениями.")
        return None
    if not np.all(np.isin(cases, ["NC", "SC"])):
        print("Функция case_star должна возвращать строки 'NC' или 'SC'.")
        return None
    fig, ax = plt.subplots(figsize=(10, 4.6), constrained_layout=True)
    factor = V0 * 1000
    ax.plot(di, factor * nc, color=ACCENT, linestyle="--", alpha=0.55,
            label=r"Формула $J_{\rm NC}$ на всём интервале")
    ax.plot(di, factor * sc, color=SECOND, linestyle="--", alpha=0.55,
            label=r"Формула $J_s$ на всём интервале")
    for name, values, color in (("NC", nc, ACCENT), ("SC", sc, SECOND)):
        valid = np.where(cases == name, values * factor, np.nan)
        ax.plot(di, valid, color=color, linewidth=2.5, label=f"{name}: применимый алгоритм")
    ax.set(xlabel=r"Взаимный угол $\Delta i$, рад",
           ylabel=r"Характеристическая скорость $\Delta V_\Sigma$, м/с",
           title="Задание 4*. Области применимости и стоимость перехода")
    ax.grid(alpha=0.22)
    ax.legend(loc="best")
    return fig, ax


def check_star(nc_cost_function, sc_cost_function, case_function):
    """Численная самопроверка стоимостей и классификации на заданном интервале."""
    if not all(callable(f) for f in (nc_cost_function, sc_cost_function, case_function)):
        print("Задание 4*: определите три скалярные функции.")
        return None
    boundary = np.sqrt(3) * abs(DEY_3)
    sample = [DI_MIN_STAR, 0.0007, boundary, 0.0013, DI_MAX_STAR]
    checks = []
    for di in sample:
        expected_nc = np.sqrt(DEX_3**2 / 4 + DEY_3**2 + di**2)
        expected_sc = 0.5 * np.hypot(DEX_3, abs(DEY_3) + np.sqrt(3) * di)
        expected_case = "NC" if di >= boundary else "SC"
        try:
            nc = nc_cost_function(di)
            sc = sc_cost_function(di)
            case = case_function(di)
            nc_array, sc_array = np.asarray(nc, dtype=float), np.asarray(sc, dtype=float)
            nc_ok = (nc is not None and nc_array.shape == () and np.isfinite(nc_array)
                     and np.isclose(float(nc_array), expected_nc, rtol=1e-6, atol=1e-10))
            sc_ok = (sc is not None and sc_array.shape == () and np.isfinite(sc_array)
                     and np.isclose(float(sc_array), expected_sc, rtol=1e-6, atol=1e-10))
            case_ok = isinstance(case, str) and case == expected_case
            label = f"Δi = {di:.10f} рад"
            checks.extend([(bool(nc_ok), "Безразмерная стоимость J_NC: " + label),
                           (bool(sc_ok), "Безразмерная стоимость J_s: " + label),
                           (case_ok, "Выбор алгоритма по областям применимости: " + label)])
        except Exception as error:
            checks.append((False, f"Δi = {di}: {type(error).__name__}: {error}"))
    return _report("Самопроверка задания 4* (контрольные примеры, не доказательство)", checks)

## Задание 2. Разные манёвры с одинаковым расходом

$$\Delta a=0{,}004,\qquad \Delta e_x=0{,}001,\qquad
  \Delta e_y=0{,}001,\qquad \Delta i=0.$$

Ось отсчёта углов фиксирована в общей плоскости орбит; компоненты
векторов эксцентриситета обеих орбит заданы относительно этой оси.

1. Определите взаимное расположение орбит и выберите алгоритм перехода.
   Объясните, возможен ли переход одним импульсом.
2. По алгоритму для непересекающихся орбит рассчитайте три двухимпульсных
   перехода, задавая положение первого импульса:
   $$\varphi_1=0^\circ,\quad120^\circ,\quad240^\circ.$$
   Для каждого найдите положение второго импульса, обе трансверсальные компоненты
   и суммарную характеристическую скорость. Результаты соберите в таблицы.
3. Проверьте три граничных условия для каждого перехода.
   Углы сначала определите по модулю $2\pi$, затем запишите их с учётом
   последовательности импульсов.
4. Найдите **один базис-вектор, подтверждающий оптимальность всех трёх решений**.
   Аналитически проверьте достаточные условия на всём обороте.
5. Объясните, почему положения и величины отдельных импульсов различаются,
   а суммарная характеристическая скорость совпадает. Что геометрия годографа
   говорит о возможности выбора места первого импульса?

> **Подсказка.** Сохраняйте знаки компонент. Угол второго импульса
> восстанавливайте по синусу и косинусу с помощью `np.arctan2`.
> Постоянные местные компоненты базис-вектора не означают постоянного
> направления в неподвижной системе координат.

**Классификация, возможность одноимпульсного перехода и расчётные формулы:**

*Обоснуйте выбор алгоритма. Запишите формулы компонент и второго угла,
которые используете для трёх вариантов.*

In [ ]:
# Три строки — три разных перехода. В каждом переходе два импульса.
phi_2 = None                  # массив (3, 2), углы в [0, 2π), рад
phi_2_chronological = None    # массив (3, 2), φ₂ > φ₁ с добавлением 2π при необходимости
# Компоненты делятся на V0, последний индекс — r,t,z.
dv_hat_2 = None               # массив (3, 2, 3)

In [ ]:
if phi_2 is not None and dv_hat_2 is not None and phi_2_chronological is not None:
    for k in range(3):
        impulse_table(phi_2_chronological[k], dv_hat_2[k],
                      f"Переход {k + 1}: первый импульс при {np.degrees(FIRST_ANGLES_2[k]):.0f}°",
                      together=True)
        boundary_table(phi_2[k], dv_hat_2[k], TARGET_2)
        print()
else:
    print("Сначала заполните phi_2, dv_hat_2 и phi_2_chronological.")

**Общий базис-вектор и доказательство оптимальности:**

*Выберите множители, запишите компоненты базис-вектора и проверьте его
совпадение с направлениями всех шести импульсов. Докажите ограничение
модуля на всём обороте. Сформулируйте вывод о неединственности решения.*

In [ ]:
lambdas_2 = None    # np.array([lambda_1, lambda_2, lambda_3, lambda_4, lambda_5])

In [ ]:
check_task2(phi_2, dv_hat_2, lambdas_2, phi_2_chronological)

## Задание 3. Узловой переход и проверка его оптимальности

$$\Delta a=0{,}001,\qquad \Delta e_x=0{,}0024,\qquad
  \Delta e_y=0{,}0006,\qquad \Delta i=0{,}0018\ \text{рад}.$$

Ось $X$ направлена к восходящему узлу конечной плоскости относительно
начальной; $\Delta e_x,\Delta e_y$ заданы в этих узловых осях.

1. Проверьте применимость узлового алгоритма (NC).
2. Найдите положения импульсов, их модули и все три компоненты каждого импульса.
   Рассчитайте суммарную характеристическую скорость. Первым нумеруйте импульс
   в восходящем узле, вторым — в нисходящем.
3. Подстановкой проверьте **все пять граничных условий** задачи перехода.
4. Определите множители $\lambda_1,\ldots,\lambda_5$ и запишите соответствующий
   базис-вектор.
5. Постройте его пространственный годограф с единичной сферой и график
   $\|\boldsymbol s(\varphi)\|$ на одном обороте. Отметьте положения импульсов.
6. Аналитически докажите $\|\boldsymbol s(\varphi)\|\leq1$ при всех $\varphi$
   и проверьте согласованность направлений базис-вектора и импульсов.
   Сформулируйте вывод об оптимальности в околокруговом приближении.
7. Коротко объясните, почему противоположные направления импульсов в местных
   орбитальных осях допускают разные модули. Как изменение $\Delta a$ внутри
   области применимости влияет на распределение импульсов, их направления и суммарную величину?

Используйте базис-вектор задачи перехода:
$$\begin{aligned}
s_r&=\lambda_3\sin\varphi-\lambda_2\cos\varphi,\\
s_t&=2\lambda_1+2\lambda_3\cos\varphi+2\lambda_2\sin\varphi,\\
s_z&=\lambda_4\sin\varphi+\lambda_5\cos\varphi.
\end{aligned}$$

> **Подсказка.** Для узлового случая $\lambda_1=0$.
> Используйте единичное направление первого импульса и стационарность
> квадрата модуля базис-вектора в точке касания сферы.
> Для доказательства запишите $\boldsymbol s(\varphi)$ через
> $\boldsymbol s(0)\cos\varphi+\boldsymbol s'(0)\sin\varphi$.
> Одного графика для ограничения между узлами сетки недостаточно.

**Область применимости, направления, модули и компоненты импульсов:**

*Приведите условия применимости, расчётные формулы и проверку пяти
граничных условий. Отличайте неотрицательные модули от знаковых компонент.*

In [ ]:
phi_3 = None       # два угла, рад: восходящий узел, затем нисходящий
dv_hat_3 = None    # np.array([[Vr_hat_1, Vt_hat_1, Vz_hat_1],
                   #           [Vr_hat_2, Vt_hat_2, Vz_hat_2]])

In [ ]:
if phi_3 is not None and dv_hat_3 is not None:
    impulse_table(phi_3, dv_hat_3, "Узловой двухимпульсный переход", together=True)
    boundary_table(phi_3, dv_hat_3, TARGET_3)
else:
    print("Сначала заполните phi_3 и dv_hat_3.")

**Определение множителей:**

*Запишите условия на базис-вектор в узлах, определите множители и проверьте
направления обоих импульсов. Покажите, какое условие позволяет найти оставшийся множитель.*

In [ ]:
lambdas_3 = None   # np.array([lambda_1, lambda_2, lambda_3, lambda_4, lambda_5])

In [ ]:
if phi_3 is not None and lambdas_3 is not None:
    fig, axes = plot_primer(phi_3, lambdas_3, "Задание 3. Узловой переход")
    plt.show()
else:
    print("Сначала заполните phi_3 и lambdas_3.")

In [ ]:
check_task3(phi_3, dv_hat_3, lambdas_3)

**Аналитическое доказательство и объяснение результатов:**

*Вычислите квадрат модуля и докажите ограничение на всём обороте.
Перечислите выполненные достаточные условия. Объясните роль $\Delta a$
и различие между противоположностью направлений и равенством модулей.*

## Задание 4*. Граница между узловым и особым случаями

**Необязательное задание.** Сохраните $\Delta a,\Delta e_x,\Delta e_y$
из задания 3, а взаимный угол плоскостей изменяйте:
$$0{,}0003\leq\Delta i\leq0{,}0018\ \text{рад}.$$

1. По условиям классификации определите области применимости узлового (NC)
   и особого (SC) алгоритмов. Найдите границу между ними.
   Проверьте все условия применимости, включая ограничение на $\Delta a$.
2. Постройте на одном графике характеристические скорости по формулам обоих
   случаев. Выделите участки, на которых каждая формула применима, и постройте
   итоговую зависимость оптимальной стоимости.
3. Аналитически проверьте совпадение стоимостей на границе.
4. Продолжите выражения для множителей узлового базис-вектора за область
   применимости NC. Исследуйте его годограф при $\Delta i=0{,}0003$ рад,
   на границе и при $\Delta i=0{,}0018$ рад.
   Объясните изменение геометрии и ограничение модуля.
5. Можно ли выбирать алгоритм, просто подставляя данные в обе формулы стоимости
   и беря меньший результат? Обоснуйте ответ расчётом и условиями применимости.

Для автоматического построения графика при желании заполните скалярные функции
`nc_cost_star(di)`, `sc_cost_star(di)` и `case_star(di)`.
Первые две возвращают **безразмерные** $J_{\rm NC}$ и $J_s$, в том числе
формальное продолжение соответствующей формулы за её область применимости.
Третья возвращает строку `'NC'` или `'SC'` по условиям классификации;
на общей границе выбирайте `'NC'` согласно порядку расчёта в методичке.

`lambdas_nc_star(di)` возвращает пять множителей узлового базис-вектора
для заданного $\Delta i$, включая продолжение вне области применимости NC.
Это продолжение само по себе не утверждает оптимальность.

**Области применимости, граница и стоимости:**

*Выведите границу, проверьте остальные ограничения и покажите совпадение
стоимостей. Объясните, почему сравнение двух чисел без проверки областей
применимости может привести к ошибке.*

In [ ]:
def nc_cost_star(di):
    """Верните безразмерный J_NC при заданном скалярном di."""
    # DA_3, DEX_3, DEY_3 сохраняются; меняется только взаимный угол.
    return None


def sc_cost_star(di):
    """Верните безразмерный J_s при заданном скалярном di."""
    return None


def case_star(di):
    """Верните строку 'NC' или 'SC'; на общей границе выбирайте 'NC'."""
    return None


def lambdas_nc_star(di):
    """Верните массив [λ₁, λ₂, λ₃, λ₄, λ₅] узлового базис-вектора."""
    return None


di_boundary_star = None  # найденная граница, рад

In [ ]:
if (nc_cost_star(DI_3) is None or sc_cost_star(DI_3) is None
        or case_star(DI_3) is None):
    print("Задание 4* необязательное. Для графика сначала заполните три функции стоимости и классификации.")
else:
    result = plot_costs_star(nc_cost_star, sc_cost_star, case_star)
    if result is not None:
        plt.show()

In [ ]:
if di_boundary_star is not None and lambdas_nc_star(DI_3) is not None:
    di_examples_star = np.array([DI_MIN_STAR, di_boundary_star, DI_MAX_STAR])
    lambda_examples_star = [lambdas_nc_star(float(di)) for di in di_examples_star]
    if any(l is None for l in lambda_examples_star):
        print("Заполните lambdas_nc_star для всех трёх значений Δi, включая продолжение вне NC.")
    else:
        for di, lambdas in zip(di_examples_star, lambda_examples_star):
            fig, axes = plot_primer(np.array([0.0, np.pi]), lambdas,
                                    f"Задание 4*. Узловой годограф: Δi = {di:.7f} рад")
            plt.show()
else:
    print("Для сравнения трёх годографов заполните lambdas_nc_star и di_boundary_star.")

In [ ]:
if (nc_cost_star(DI_3) is None or sc_cost_star(DI_3) is None
        or case_star(DI_3) is None):
    print("Для необязательной самопроверки сначала заполните три функции стоимости и классификации.")
else:
    check_star(nc_cost_star, sc_cost_star, case_star)

**Геометрия годографа и вывод:**

*Опишите форму узлового годографа и положение относительно единичной сферы
в трёх выбранных случаях. Свяжите это с применимостью алгоритма и выбором
оптимальной стоимости.*

Чекеры проверяют численные граничные условия, направления и достаточную
глобальную оценку модуля для найденных множителей. В центральном узловом случае
оценка модуля вычисляется по спектру матрицы Грама и является точным максимумом
на полном обороте. Для смещённого годографа используется верхняя оценка,
которая в общем случае может превышать точный максимум.
Графики строятся по угловой сетке и служат иллюстрацией.
Эти проверки **не заменяют аналитических доказательств**.

## Что представить

Сдайте этот ноутбук с ответами на оба вопроса задания 1,
полными решениями заданий 2 и 3 и, по желанию, задания 4*:
расчётными формулами, таблицами импульсов, проверками граничных условий,
годографом и графиком модуля базис-вектора, обоснованием оптимальности.
Для задания 4* добавьте график стоимостей и сравнение трёх годографов.
Использование чекеров и сохранение результатов их вызова не требуются.

Перед отправкой перезапустите ядро, выполните ячейки решения по порядку и сохраните
ноутбук с результатами. Название файла: **`hw_5_name_surname.ipynb`**;
вместо `name_surname` укажите свои имя и фамилию латиницей.